# Sesiones 12 y 13 · Centro: moda, mediana y media

**Antes de empezar:** Entorno de ejecución → Cambiar tipo de entorno de ejecución
→ **R**, y después Archivo → Guardar una copia en Drive.

## 1 · Los hogares de México

Encuesta Nacional de Ingresos y Gastos de los Hogares (ENIGH) 2024, del INEGI.
Cada renglón es un hogar, sin importar cuántas personas vivan en él. Las columnas
`ingreso` y `gasto` están **en pesos por trimestre**.

El archivo está comprimido y `read_csv()` lo abre igual.

In [ ]:
library(tidyverse)
source("https://raw.githubusercontent.com/cjjmdata/analisis_datos_i/main/R/colores.R")   # colores del curso: col_media, col_mediana, col_moda

hogares <- read_csv("https://raw.githubusercontent.com/cjjmdata/analisis_datos_i/main/datos/enigh2024_hogares.csv.gz", show_col_types = FALSE)

glimpse(hogares)

La columna `factor` dice a cuántos hogares del país representa cada hogar
visitado. Las cifras calculadas sin factor describen a los hogares visitados; las
que usan el factor describen a los hogares del país.

Antes de seguir, anota tu apuesta: un hogar típico en México, ¿cuánto recibe al
mes?

## 2 · La moda

El valor más frecuente. Es la medida de centro que admite cualquier variable
categórica.

Antes de correr la celda: ¿cuál es la escolaridad más frecuente entre los jefes y
jefas de hogar?

In [ ]:
escolaridad <- count(hogares, educa_jefe, sort = TRUE)

escolaridad

In [ ]:
ggplot(escolaridad, aes(x = n, y = fct_reorder(educa_jefe, n))) +
  geom_col(fill = col_moda, alpha = 0.85) +
  geom_text(aes(label = n), hjust = -0.15) +
  scale_x_continuous(expand = expansion(mult = c(0, 0.18))) +
  labs(x = "Hogares visitados", y = NULL)

Con pocos valores posibles, como el número de integrantes, la moda es un valor
exacto. ¿Cuántos integrantes tiene el hogar más frecuente?

In [ ]:
count(hogares, integrantes, sort = TRUE)

### Con una variable continua

¿Cuántos hogares crees que coinciden en el mismo ingreso exacto?

In [ ]:
length(unique(hogares$ingreso))

sort(table(hogares$ingreso), decreasing = TRUE)[1:3]

El valor exacto que más se repite señala una coincidencia. Para encontrar dónde se
concentran los datos, primero se agrupan en intervalos: el intervalo con más
observaciones es la **clase modal**, y su punto medio es la moda estimada.

In [ ]:
cortes <- seq(0, 200000, by = 10000)
conteo <- table(cut(hogares$ingreso, breaks = cortes, right = FALSE, dig.lab = 10))

sort(conteo, decreasing = TRUE)[1:3]

In [ ]:
i <- which.max(conteo)

moda_clase <- (cortes[i] + cortes[i + 1]) / 2   # punto medio del intervalo
moda_clase

### El ancho decide la moda

Cambia `ancho` a 5000 y después a 20000, y vuelve a correr la celda.

In [ ]:
ancho <- 10000

cortes_prueba <- seq(0, 300000, by = ancho)
conteo_prueba <- table(cut(hogares$ingreso, breaks = cortes_prueba, right = FALSE, dig.lab = 10))
j <- which.max(conteo_prueba)

(cortes_prueba[j] + cortes_prueba[j + 1]) / 2

Tres anchos, tres modas, y las tres bien calculadas. La moda de una variable
continua se **estima**, y el resultado depende del ancho del intervalo, igual que
la forma del histograma depende de su `binwidth`. Una moda estimada se reporta
junto con el ancho que la produjo.

### Un pico o varios

La estatura de la encuesta del grupo, por género.

In [ ]:
url_grupo <- paste0("https://docs.google.com/spreadsheets/d/e/2PACX-1vTTKC57eWZVIQ9lwhoR5nV",
                    "qYM4kgi5zA9yifa-YStdfdJwNe7ATs0p-TUCUwvjfcWmHmvsZDEK8VX4I/pub?gid=1326008067&single=true&output=csv")

estaturas <- read_csv(url_grupo, show_col_types = FALSE) |>
  select(genero = `Género`, estatura = `Estatura, en metros`) |>
  filter(!is.na(estatura))

ggplot(estaturas, aes(x = estatura)) +
  geom_freqpoly(aes(color = "Todo el grupo"), binwidth = 0.02, boundary = 1.5,
                linewidth = 1.4) +
  geom_freqpoly(aes(color = genero), binwidth = 0.02, boundary = 1.5,
                linewidth = 0.9) +
  scale_color_manual(values = c(`Todo el grupo` = TINTA,
                                Femenino  = col_grupos[1],
                                Masculino = col_grupos[2]),
                     name = NULL) +
  labs(x = "Estatura, en metros", y = "Estudiantes")

¿Dónde estaría la moda de la estatura de todo el grupo?

Una distribución con un solo pico es **unimodal**; con dos, **bimodal**; con más,
**multimodal**. La línea oscura suma las otras dos en cada intervalo, y por eso
tiene dos picos. Dos picos suelen señalar dos poblaciones mezcladas, y se separan
antes de resumir.

## 3 · La mediana

Deja la mitad de los hogares debajo y la mitad arriba. Es el segundo cuartil, con
otro uso.

In [ ]:
median(hogares$ingreso)

quantile(hogares$ingreso, 0.50)

El hogar con mayor ingreso de la base recibe más de 17 millones de pesos en el
trimestre. Antes de correr las celdas: si sale de la base junto con el resto del
1% más alto, ¿cuánto cambia la mediana?

In [ ]:
quitados <- filter(hogares, ingreso <= quantile(ingreso, 0.99))

nrow(hogares) - nrow(quitados)     # hogares que salieron

max(hogares$ingreso)
max(quitados$ingreso)

In [ ]:
median(hogares$ingreso)
median(quitados$ingreso)

In [ ]:
comparacion <- bind_rows(
  mutate(hogares,  base = "Todos los hogares"),
  mutate(quitados, base = "Sin el 1% más alto")
)

ggplot(comparacion, aes(x = ingreso, y = base)) +
  geom_boxplot(fill = col_datos, alpha = 0.5, outlier.alpha = 0.12) +
  geom_vline(xintercept = median(hogares$ingreso), color = col_mediana, linewidth = 1) +
  coord_cartesian(xlim = c(0, 300000)) +
  labs(x = "Ingreso trimestral del hogar, en pesos", y = NULL)

La mediana se calcula por posición, y la posición de en medio apenas se recorre.
El eje llega hasta 300,000: `coord_cartesian()` recorta la vista, no los datos.

## 4 · La media

Suma todos los valores y reparte el total entre el número de observaciones:

$$\bar{x} = \frac{\sum x_i}{n}$$

Es la única de las tres medidas que usa todos los valores, así que cualquier
valor que entra o sale la mueve.

In [ ]:
sum(hogares$ingreso) / length(hogares$ingreso)

mean(hogares$ingreso)

In [ ]:
mean(hogares$ingreso)
mean(quitados$ingreso)

Las tres medidas sobre el mismo histograma. Verde la moda, azul la mediana, rojo
la media: los tres colores se usan igual en todo el curso.

In [ ]:
ggplot(filter(hogares, ingreso <= 200000), aes(x = ingreso)) +
  geom_histogram(binwidth = 10000, fill = col_datos, color = "white", boundary = 0) +
  geom_vline(xintercept = moda_clase, color = col_moda, linewidth = 1.2) +
  geom_vline(xintercept = median(hogares$ingreso), color = col_mediana, linewidth = 1.2) +
  geom_vline(xintercept = mean(hogares$ingreso), color = col_media, linewidth = 1.2) +
  labs(x = "Ingreso trimestral del hogar, en pesos", y = "Hogares visitados")

¿Por qué quedan en ese orden? Corre la celda: ¿qué porcentaje de los hogares
visitados recibe menos que la media?

In [ ]:
mean(hogares$ingreso < mean(hogares$ingreso)) * 100

La media reparte el total entre todos los hogares, así que un ingreso muy alto
sube el promedio sin cambiar la situación de ningún otro hogar. Con una cola
larga a la derecha, la media describe el reparto del total y la mediana describe
al hogar de en medio.

## 5 · La media ponderada

Un curso con tres componentes que pesan distinto. La media simple trata igual a
los tres; la media ponderada multiplica cada calificación por su peso:

$$\bar{x}_w = \frac{\sum w_i\,x_i}{\sum w_i}$$

**Cambia los pesos y las calificaciones por los de tu curso** y vuelve a correr
la celda. ¿Cuánto se separa tu media ponderada de la simple?

In [ ]:
curso <- tibble(
  componente   = c("Primer parcial", "Segundo parcial", "Portafolio"),
  calificacion = c(70, 80, 100),
  peso         = c(0.20, 0.30, 0.50)
)

mean(curso$calificacion)

sum(curso$peso * curso$calificacion) / sum(curso$peso)

weighted.mean(curso$calificacion, curso$peso)

### De la muestra a la población

La **población** son los hogares que había en México en 2024. La **muestra** son
los que la encuesta visitó. Cada hogar visitado representa a un número distinto
de hogares del país, y ese número es su `factor`.

In [ ]:
nrow(hogares)          # la muestra: hogares visitados

sum(hogares$factor)    # la población: hogares del país

range(hogares$factor)

La muestra se eligió en dos etapas. Primero las zonas del país se agrupan en 1,175
estratos según entidad, tamaño de localidad y nivel socioeconómico; en cada
estrato se sortean conglomerados de 80 a 160 viviendas, y dentro de cada
conglomerado se sortean las viviendas que se visitan. Es una muestra
**probabilística**: cada vivienda tiene una probabilidad conocida de ser elegida.

El factor es el inverso de esa probabilidad. Si una vivienda tenía una
probabilidad de 1 en 400 de ser elegida, se cuenta por 400. Después se ajusta por
las viviendas que no respondieron y para que el total coincida con la estimación
de población del INEGI.

Fuente: INEGI (2025), *ENIGH 2024. Nueva serie. Diseño muestral*, pp. 1 a 8.

In [ ]:
mean(hogares$ingreso)                           # los hogares visitados

weighted.mean(hogares$ingreso, hogares$factor)  # los hogares del país

La cifra ponderada, 77,864 pesos trimestrales, es la que publica el INEGI.

Un **decil de ingreso** parte al país en diez grupos con el mismo número de
hogares: se ordenan por ingreso, se acumula a cuántos hogares representan, y se
corta cada vez que se junta otro 10%.

In [ ]:
deciles <- hogares |>
  arrange(ingreso) |>
  mutate(decil = pmin(ceiling(cumsum(factor) / sum(factor) * 10), 10)) |>
  summarise(visitados = n(),
            representa = sum(factor),
            factor_promedio = round(mean(factor)), .by = decil)

deciles

In [ ]:
ggplot(deciles, aes(x = decil, y = visitados)) +
  geom_col(fill = col_datos, width = 0.75) +
  geom_text(aes(label = visitados), vjust = -0.5) +
  scale_x_continuous(breaks = 1:10) +
  labs(x = "Decil de ingreso, del más bajo al más alto", y = "Hogares visitados")

Los diez deciles representan el mismo número de hogares del país, pero para
juntar el decil más bajo hizo falta visitar más hogares que para el más alto. La
muestra tiene proporcionalmente más hogares de ingreso bajo que el país, y el
factor les quita peso.

### La mediana del país

Usa el mismo corte que los deciles, al 50%.

In [ ]:
mediana_pais <- hogares |>
  arrange(ingreso) |>
  mutate(acumulado = cumsum(factor) / sum(factor)) |>
  filter(acumulado >= 0.5) |>
  slice(1) |>
  pull(ingreso)

mediana_pais
median(hogares$ingreso)    # la de los hogares visitados

Cualquier cifra que hable del país se calcula con el factor: la media, la
mediana, los porcentajes y los conteos.

## 6 · El perfil de Oaxaca

Antes de correr la celda: ordenadas las 32 entidades de menor a mayor ingreso
promedio por hogar, ¿en qué lugar queda Oaxaca?

In [ ]:
hogares |>
  summarise(ingreso = weighted.mean(ingreso, factor), .by = entidad) |>
  arrange(ingreso)

In [ ]:
oaxaca <- filter(hogares, entidad == "Oaxaca")

mediana_con_factor <- function(x, w) {
  o <- order(x)
  x[o][which(cumsum(w[o]) / sum(w) >= 0.5)[1]]
}

weighted.mean(oaxaca$ingreso, oaxaca$factor)
mediana_con_factor(oaxaca$ingreso, oaxaca$factor)

weighted.mean(hogares$ingreso, hogares$factor)
mediana_pais

Compara las dos proporciones: el promedio de Oaxaca contra el nacional, y la
mediana de Oaxaca contra la nacional. Escribe en tu cuaderno qué dice que sean
parecidas.

Cambia `"Oaxaca"` por otra entidad y vuelve a correr las celdas.

### Tu apuesta, revisada

Las tres medidas de centro para los hogares del país, al trimestre y al mes. ¿A
cuál se parece la cifra que anotaste al principio? ¿Qué pregunta estabas
contestando?

In [ ]:
cortes_p <- seq(0, 200000, by = 10000)
clase_p  <- cut(hogares$ingreso, breaks = cortes_p, right = FALSE)
k <- which.max(tapply(hogares$factor, clase_p, sum))   # clase modal con factor

tibble(
  medida    = c("Moda (clase modal)", "Mediana", "Media"),
  trimestre = c((cortes_p[k] + cortes_p[k + 1]) / 2,
                mediana_pais,
                weighted.mean(hogares$ingreso, hogares$factor))
) |>
  mutate(mes = trimestre / 3)

## 7 · Otra base: el PIB de quince países

El Banco Mundial publica el PIB per cápita, en dólares corrientes, y la población
de cada país.

In [ ]:
paises <- read_csv("https://raw.githubusercontent.com/cjjmdata/analisis_datos_i/main/datos/banco_mundial.csv", show_col_types = FALSE)

ultimo <- paises |>
  filter(anio == max(anio)) |>
  select(pais, pib_per_capita, poblacion) |>
  arrange(desc(pib_per_capita))

ultimo |>
  mutate(pib_per_capita = round(pib_per_capita),
         millones_hab = round(poblacion / 1e6, 1),
         .keep = "unused") |>
  print(n = 15)

Los cálculos usan las cifras sin redondear; el redondeo es solo para leer la
tabla.

Antes de correr la celda: ¿cuál es el PIB per cápita promedio de estos países?

In [ ]:
nrow(ultimo)

mean(ultimo$pib_per_capita)

Ese promedio cuenta igual a cada país. ¿Describe a las personas que viven en
ellos? En la gráfica, el área de cada círculo es la población; la línea punteada es
la media simple y la continua, la media ponderada por población.

In [ ]:
simple_pib <- mean(ultimo$pib_per_capita)
pond_pib   <- weighted.mean(ultimo$pib_per_capita, ultimo$poblacion)

ggplot(ultimo, aes(x = pib_per_capita, y = fct_reorder(pais, pib_per_capita))) +
  geom_vline(xintercept = simple_pib, color = col_media, linetype = "dashed") +
  geom_vline(xintercept = pond_pib, color = col_media) +
  geom_point(aes(size = poblacion), color = TEAL_500, alpha = 0.7) +
  scale_size_area(max_size = 16, guide = "none") +
  scale_x_continuous(labels = scales::comma,
                     expand = expansion(mult = c(0.12, 0.05))) +
  labs(x = "PIB per cápita, dólares", y = NULL)

Dos preguntas, dos promedios:

| Pregunta | Unidad de observación | Cálculo |
|---|---|---|
| ¿Cuánto produce por habitante el país típico? | El país | `mean()` |
| ¿Cuánto produce por habitante el conjunto? | La persona | `weighted.mean()` con la población |

La media ponderada por población es el PIB total de los quince dividido entre su
población total. Compruébalo:

In [ ]:
sum(ultimo$pib_per_capita * ultimo$poblacion) /   # PIB total de los quince
  sum(ultimo$poblacion)                             # entre su población total

pond_pib

Los dos promedios están bien calculados y cada uno contesta su pregunta. La
pregunta decide la unidad de observación, y la unidad decide el peso.

### Lo que el PIB per cápita no dice

El PIB per cápita reparte la producción total en partes iguales entre todos los
habitantes. Leerlo como lo que recibe cada persona supone que todos reciben la
misma parte, y nada en el número dice cómo se reparte dentro de cada país. Además
mide producción, no el ingreso que llega a los hogares.

¿Qué porcentaje de las personas vive en países por debajo del promedio ponderado?

In [ ]:
sum(ultimo$poblacion[ultimo$pib_per_capita < pond_pib]) /
  sum(ultimo$poblacion) * 100

## Tarea

Sobre **tu serie del portafolio**:

1. La media y la mediana de tu variable numérica, con sus unidades.
2. Cuál de las dos describe mejor tu serie, con una razón que salga de la forma
   de tu histograma.
3. El porcentaje de tus observaciones que queda debajo de la media.
4. Si tu base trae alguna columna que sirva de peso, la media ponderada y la
   diferencia contra la simple. Si no la trae, qué pesaría si la tuviera.
5. Una línea sobre lo que tu medida de centro no muestra.

**Guarda tu copia.**